# 8086 LLM - GGUF Exporter
This notebook merges your trained LoRA adapters into the Qwen2.5-7B base model, and compiles it into a highly compressed `.gguf` file for Ollama.

In [ ]:
!pip install -q -U transformers peft accelerate
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp && pip install -r requirements.txt

## 1. Upload Adapters
**STOP!** Before running the next cell, upload your `adapters.zip` file to the Colab sidebar and run this cell to extract them.

In [ ]:
import os
import zipfile

if not os.path.exists("adapters.zip"):
    raise FileNotFoundError("Please upload adapters.zip first!")

with zipfile.ZipFile("adapters.zip", 'r') as zip_ref:
    zip_ref.extractall("adapters")
print("Adapters extracted successfully!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "Qwen/Qwen2.5-7B"

print("Loading base model (This splits across GPU and CPU RAM to prevent OOM)...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Applying LoRA...")
model = PeftModel.from_pretrained(base_model, "adapters")

print("Merging weights...")
merged_model = model.merge_and_unload()

print("Saving merged model...")
merged_model.save_pretrained("merged_model", max_shard_size="2GB")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.save_pretrained("merged_model")
print("Merge complete!")

In [ ]:
print("Converting to 4-bit GGUF format... (This takes a few minutes)")
!python llama.cpp/convert_hf_to_gguf.py merged_model --outfile 8086-model-Q4_K_M.gguf --outtype q4_k_m

In [ ]:
from google.colab import files
print("Downloading GGUF file to your computer...")
files.download("8086-model-Q4_K_M.gguf")